## Imports

In [ ]:
%pip -q install -U langchain langchain-text-splitters langchain-community langchain-experimental bs4 sentence-transformers
%pip -q install -U langchain-core
%pip -q install -U langchain-huggingface huggingface_hub
%pip -q install -U datasets ragas

import getpass
import os
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from ragas import evaluate
import re
import ast
import langchain
import time

## Hugging Face (Inference API)

In [ ]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""
if not os.environ.get("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass(
        "Enter Hugging Face token (HUGGINGFACEHUB_API_TOKEN): "
    )

# Allowed model list
HF_MODELS = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-72B-Instruct",
    "mistralai/Mistral-Nemo-Instruct-2407",
    "deepseek-ai/DeepSeek-R1",
]

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", HF_MODELS[0])
if HF_MODEL_ID not in HF_MODELS:
    raise ValueError(f"HF_MODEL_ID must be one of {HF_MODELS}. Got: {HF_MODEL_ID!r}")

HF_PROVIDER = os.environ.get("HF_PROVIDER", "auto")
HF_MAX_NEW_TOKENS = int(os.environ.get("HF_MAX_NEW_TOKENS", "4096"))
HF_TEMPERATURE = float(os.environ.get("HF_TEMPERATURE", "0.5"))
HF_TOP_P = float(os.environ.get("HF_TOP_P", "0.95"))

# Endpoint for agent: text-generation + ChatHuggingFace wrapper
llm_agent = HuggingFaceEndpoint(
    repo_id=HF_MODEL_ID,
    task="text-generation",
    provider=HF_PROVIDER,
    max_new_tokens=HF_MAX_NEW_TOKENS,
    temperature=HF_TEMPERATURE,
    top_p=HF_TOP_P,
    return_full_text=False,
)
model = ChatHuggingFace(llm=llm_agent)

# Endpoint for RAGAS evaluation: conversational task (some providers like Novita require this)
llm = HuggingFaceEndpoint(
    repo_id=HF_MODEL_ID,
    task="conversational",
    provider=HF_PROVIDER,
    max_new_tokens=HF_MAX_NEW_TOKENS,
    temperature=HF_TEMPERATURE,
    top_p=HF_TOP_P,
    return_full_text=False,
)

print(f"Using Hugging Face model: {HF_MODEL_ID}")
print(f"HF_PROVIDER={HF_PROVIDER}")
print(f"HF_MAX_NEW_TOKENS={HF_MAX_NEW_TOKENS}")

Using Hugging Face model: meta-llama/Llama-3.1-8B-Instruct
HF_PROVIDER=auto
HF_MAX_NEW_TOKENS=4096


## Embeddings

In [ ]:
# Embeddings (local)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


#### Data and Chunks

In [ ]:
csv_path = Path("/fashion.csv")

loader = CSVLoader(file_path=str(csv_path), encoding="utf-8")
docs = loader.load()

all_splits = docs

# Index chunks
_ = vector_store.add_documents(documents=all_splits)

## RAG Agent

In [5]:
# --- RAG chain via dynamic prompt (middleware) ---
# This runs retrieval automatically on every user message and injects the
# retrieved content into the model prompt.
@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    last_msg = request.state["messages"][-1]

    # Be robust across different message object shapes
    last_query = getattr(last_msg, "text", None) or getattr(last_msg, "content", "")

    retrieved_docs = vector_store.similarity_search(last_query, k=3)
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful shopping assistant for a fashion catalog. Use the following product catalog context to answer. "
        "If the answer isn't in the context, say you don't know.\n\n"
        "Output format (IMPORTANT):\n"
        "- Return your final answer as GitHub-flavored Markdown.\n"
        "- When you list products, for each product include: ProductTitle, ProductId, Size (if present), and ImageURL.\n"
        "- Also embed the image preview using Markdown image syntax on its own line: ![ProductTitle](ImageURL)\n\n"
        "CATALOG CONTEXT:\n"
        f"{docs_content}"
    )

    return system_message

agent = create_agent(model, tools=[], middleware=[prompt_with_context])

#### Query

In [ ]:
query = "hi, do you have girls pink top?"

resp = agent.invoke({"messages": [{"role": "user", "content": query}]})
md = resp["messages"][-1].content

display(Markdown(md))

## Evaluation

### Generation Evaluation

#### Create Test Examples

In [ ]:
# Hard-coded examples for evaluation
examples = [
    {
        "query": "Do you have any pink tops for girls?",
        "answer": "Yes, we have pink tops available in the fashion catalog"
    },
    {
        "query": "What sizes are available for girls clothing?",
        "answer": "Various sizes are available depending on the product"
    },
    {
        "query": "Do you have girls shoes?",
        "answer": "Yes, we have Disney Kids Princess Heart Pink Casual Shoes in the fashion catalog"
    }
]

#### Generate Additional Examples using LLM

In [ ]:
# Create example generation prompt
example_gen_prompt = PromptTemplate.from_template(
    """Given the following document, generate a question and answer pair that could be asked about it.

Document:
{doc}

Please respond in JSON format with "query" and "answer" keys."""
)

# Generate new examples from first few documents WITH RATE LIMITING
new_examples = []
for i, doc in enumerate(docs[:3]):
    try:
        if i > 0:
            time.sleep(3)  # Wait 3 seconds between requests
        chain = example_gen_prompt | model | JsonOutputParser()
        result = chain.invoke({"doc": doc.page_content})
        new_examples.append(result)
        print(f"Generated example {i+1}")
    except Exception as e:
        print(f"Skipping example generation due to: {e}")

# Combine with hard-coded examples
examples += new_examples

print(f"Total examples: {len(examples)}")
if new_examples:
    print(f"First generated example: {new_examples[0]}")

In [ ]:
# Generate queries and ground truths for RAGAS evaluation
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json

# Create RAGAS-specific example generation prompt
ragas_gen_prompt = PromptTemplate.from_template(
    """You are generating evaluation examples for a RAG (Retrieval Augmented Generation) system
that answers questions about a fashion product catalog.

Based on the following document snippet from the catalog, generate a realistic user query and
its corresponding ground truth answer.

Document:
{doc}

Requirements:
1. Generate a natural, conversational query that a real customer might ask
2. The ground truth answer should be accurate and based ONLY on the information in the document
3. The answer should include specific product details like ProductTitle, ProductId, Size (if available), and ImageURL
4. Format as JSON with keys: "query" (string) and "ground_truth" (string)

Respond ONLY with valid JSON, no additional text."""
)

# Generate RAGAS evaluation examples from documents WITH RATE LIMITING
ragas_examples = []
sample_docs = docs[:10]  # Use first 10 documents to generate diverse examples

print("Generating RAGAS evaluation examples...")
for i, doc in enumerate(sample_docs):
    try:
        if i > 0:
            time.sleep(3)  # Wait 3 seconds between requests to avoid rate limiting

        chain = ragas_gen_prompt | model | JsonOutputParser()
        result = chain.invoke({"doc": doc.page_content})

        # Add retrieved contexts (the document itself) for RAGAS evaluation
        ragas_example = {
            "query": result["query"],
            "ground_truth": result["ground_truth"],
            "contexts": [doc.page_content],  # For RAGAS, we include the context
            "doc_metadata": doc.metadata if hasattr(doc, 'metadata') else {}
        }
        ragas_examples.append(ragas_example)
        print(f"Generated RAGAS example {i+1}/{len(sample_docs)}: {result['query'][:50]}...")
    except Exception as e:
        print(f"Error generating example {i+1}: {e}")
        continue

print(f"\nTotal RAGAS examples generated: {len(ragas_examples)}")
if ragas_examples:
    print(f"\nExample RAGAS evaluation entry:")
    print(f"Query: {ragas_examples[0]['query']}")
    print(f"Ground Truth: {ragas_examples[0]['ground_truth'][:100]}...")
    print(f"Contexts: {len(ragas_examples[0]['contexts'])} document(s)")


#### Run Agent on Examples

In [ ]:
# Test agent on first example
test_response = agent.invoke({"messages": [{"role": "user", "content": examples[0]["query"]}]})
print(f"Query: {examples[0]['query']}")
print(f"Agent Response: {test_response['messages'][-1].content[:200]}...")

#### Manual Evaluation with Debug Mode

In [ ]:
# Enable debug mode to see internal steps
langchain.debug = True

# Run a query in debug mode
test_response = agent.invoke({"messages": [{"role": "user", "content": examples[0]["query"]}]})

# Disable debug mode
langchain.debug = False

#### LLM-Assisted Evaluation using LLM

In [ ]:
# Get predictions from agent on all examples WITH RATE LIMITING
predictions = []
for i, example in enumerate(examples):
    if i > 0:
        time.sleep(3)  # Wait 3 seconds between requests
    try:
        resp = agent.invoke({"messages": [{"role": "user", "content": example["query"]}]})
        predictions.append({
            "query": example["query"],
            "answer": example["answer"],
            "result": resp["messages"][-1].content
        })
        print(f"Processed prediction {i+1}/{len(examples)}")
    except Exception as e:
        print(f"Error on example {i+1}: {e}")
        predictions.append({
            "query": example["query"],
            "answer": example["answer"],
            "result": "Error: Rate limited or API error"
        })

print(f"Generated {len(predictions)} predictions")

In [ ]:
import time
from langchain_core.prompts import PromptTemplate

# Create evaluation prompt
eval_prompt = PromptTemplate.from_template(
    """You are an expert evaluator. Compare the predicted answer to the expected answer.

Question: {query}
Expected Answer: {answer}
Predicted Answer: {result}

Evaluate whether the predicted answer correctly addresses the question.
Respond with either "CORRECT" or "INCORRECT" followed by a brief explanation."""
)

# Evaluate predictions against ground truth WITH RATE LIMITING
graded_outputs = []
for i, pred in enumerate(predictions):
    if i > 0:
        time.sleep(3)  # Wait 3 seconds between requests
    try:
        chain = eval_prompt | model
        grade = chain.invoke({
            "query": pred["query"],
            "answer": pred["answer"],
            "result": pred["result"]
        })
        graded_outputs.append({"text": grade.content})
        print(f"Evaluated {i+1}/{len(predictions)}")
    except Exception as e:
        graded_outputs.append({"text": f"Error evaluating: {str(e)}"})
        print(f"Error evaluating {i+1}: {e}")

print(f"Evaluation complete. Graded {len(graded_outputs)} examples")

### Retrieval Evaluation

#### Modern RAGAS evaluation using Dataset and ground truths

In [ ]:
csv_ragas_examples = []
csv_path = Path("retrieval_evaluation_queries.csv")
if csv_path.exists():
    df_csv = pd.read_csv(csv_path)
    for _, row in df_csv.iterrows():
        # Parse contexts from string representation to list
        contexts_str = row["contexts"]
        try:
            # Try to evaluate the string as a Python literal (list)
            contexts = ast.literal_eval(contexts_str) if isinstance(contexts_str, str) else contexts_str
        except Exception:
            # If parsing fails, wrap in a list
            contexts = [contexts_str] if contexts_str else []

        csv_ragas_examples.append(
            {
                "query": row["query"],
                "ground_truth": row["ground_truth"],
                "contexts": contexts,
                "doc_metadata": ast.literal_eval(row["doc_metadata"]) if isinstance(row["doc_metadata"], str) else row["doc_metadata"],
            }
        )
    print(f"Loaded {len(csv_ragas_examples)} examples from {csv_path}")
    # Use CSV examples instead of generated ones
    ragas_examples = csv_ragas_examples
else:
    print(f"CSV file {csv_path} not found, using generated examples")

# --- Rate-limit / quota prevention knobs ---
EVAL_MAX_EXAMPLES = int(os.environ.get("EVAL_MAX_EXAMPLES", "4"))
REQUEST_SLEEP_SECONDS = float(os.environ.get("REQUEST_SLEEP_SECONDS", "5"))
MAX_RETRIES = int(os.environ.get("MAX_RETRIES", "6"))
RETRIEVAL_K = int(os.environ.get("RETRIEVAL_K", "2"))  # smaller k => fewer input tokens later
MAX_CONTEXT_CHARS = int(os.environ.get("MAX_CONTEXT_CHARS", "1200"))

ragas_examples = ragas_examples[:EVAL_MAX_EXAMPLES]


def _sleep_for_quota(e: Exception, attempt: int) -> float:
    """Return how long to sleep (seconds) for quota/rate errors."""
    msg = str(e)

    # If your plan/token has 0 quota, retries won't help; fail fast with guidance.
    if "limit: 0" in msg and ("RESOURCE_EXHAUSTED" in msg or "429" in msg):
        raise RuntimeError(
            "The model provider returned a quota/rate-limit error with limit 0. "
            "Check that your provider token/billing/quota is enabled for the selected model, then retry.\n\n"
            f"Original error: {msg}"
        )

    # Honor server hint: "Please retry in Xs."
    m = re.search(r"Please retry in\s+([0-9.]+)s", msg)
    if m:
        return float(m.group(1)) + 1.0

    # Exponential backoff with cap
    return min(60.0, 2.0 ** attempt)


def _invoke_agent_with_retry(query: str) -> str:
    last_err = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            resp = agent.invoke({"messages": [{"role": "user", "content": query}]})
            return resp["messages"][-1].content
        except Exception as e:
            last_err = e
            msg = str(e)
            if "RESOURCE_EXHAUSTED" in msg or "429" in msg:
                wait_s = _sleep_for_quota(e, attempt)
                print(f"Rate/quota limited. Sleeping {wait_s:.1f}s then retrying (attempt {attempt+1}/{MAX_RETRIES})...")
                time.sleep(wait_s)
                continue
            raise

    raise RuntimeError(f"Agent call failed after retries. Last error: {last_err}")


def _truncate(s: str, max_chars: int) -> str:
    if s is None:
        return ""
    s = str(s)
    return s if len(s) <= max_chars else (s[: max_chars - 20] + "\n... [truncated] ...")


Loaded 10 examples from retrieval_evaluation_queries.csv
Generating responses and retrieving contexts for RAGAS evaluation...
Evaluating 4 examples

Processed 1/4: Hey, do you have any white tops for girls from Gin...
Processed 2/4: I'm looking for a black top for my daughter. Do yo...
Processed 3/4: I'm looking for a blue casual top for my daughter....
Processed 4/4: Hi, I'm looking for a cute pink top for my daughte...

Prepared 4 examples for RAGAS evaluation


In [ ]:
print("Generating responses and retrieving contexts for RAGAS evaluation...")
print(f"Evaluating {len(ragas_examples)} examples\n")

# Generate responses and retrieve contexts for each example
evaluation_data = []
for i, example in enumerate(ragas_examples):
    try:
        if i > 0:
            time.sleep(REQUEST_SLEEP_SECONDS)  # basic pacing between requests

        query = example["query"]

        # Generate response using the agent (with retry/backoff)
        response = _invoke_agent_with_retry(query)

        # Retrieve contexts using vector store (for proper retrieval evaluation)
        retrieved_docs = vector_store.similarity_search(query, k=RETRIEVAL_K)
        retrieved_contexts = [_truncate(doc.page_content, MAX_CONTEXT_CHARS) for doc in retrieved_docs]

        evaluation_data.append(
            {
                "question": query,
                "answer": response,
                "contexts": retrieved_contexts,
                "ground_truth": example["ground_truth"],
            }
        )

        print(f"Processed {i+1}/{len(ragas_examples)}: {query[:50]}...")
    except Exception as e:
        print(f"Error processing example {i+1}: {e}")
        continue

print(f"\nPrepared {len(evaluation_data)} examples for RAGAS evaluation")

In [ ]:
print(evaluation_data)

#### View Evaluation Results

In [ ]:
# RETRIEVAL EVALUATION WITH STANDARD METRICS
# NOTE: Full RAGAS with HuggingFace serverless inference has provider/task compatibility issues

print("Computing retrieval evaluation metrics...")

results_list = []
all_recalls = []
all_precisions = []
all_avg_precisions = []
all_reciprocal_ranks = []

for item in evaluation_data:
    question = item["question"]
    answer = item["answer"]
    contexts = item["contexts"]
    ground_truth = item["ground_truth"]

    # Extract key information from ground truth (ProductId, product name, etc.)
    gt_lower = ground_truth.lower()
    gt_words = set(gt_lower.split())

    # Determine which contexts are relevant by checking overlap with ground truth
    relevant_contexts = []
    retrieved_contexts = []

    for idx, context in enumerate(contexts):
        context_lower = context.lower()
        context_words = set(context_lower.split())

        # A context is considered relevant if it has significant overlap with ground truth
        # or contains the same ProductId
        overlap = len(gt_words.intersection(context_words))
        overlap_ratio = overlap / len(gt_words) if len(gt_words) > 0 else 0.0

        retrieved_contexts.append(idx)

        # Consider relevant if overlap ratio > threshold or contains matching ProductId
        if overlap_ratio > 0.15:  # 15% overlap threshold
            relevant_contexts.append(idx)

    # Calculate metrics
    num_relevant = len(relevant_contexts)
    num_retrieved = len(retrieved_contexts)
    num_relevant_retrieved = len(relevant_contexts)  # Since we only retrieve what's relevant

    # Recall: proportion of relevant documents that were retrieved
    recall = num_relevant_retrieved / num_relevant if num_relevant > 0 else 0.0

    # Precision: proportion of retrieved documents that are relevant
    precision = num_relevant_retrieved / num_retrieved if num_retrieved > 0 else 0.0

    # Average Precision (AP): precision at each relevant document position
    if num_relevant > 0:
        precisions_at_k = []
        relevant_count = 0
        for k, idx in enumerate(retrieved_contexts, 1):
            if idx in relevant_contexts:
                relevant_count += 1
                precisions_at_k.append(relevant_count / k)
        avg_precision = sum(precisions_at_k) / num_relevant if precisions_at_k else 0.0
    else:
        avg_precision = 0.0

    # Reciprocal Rank (RR): 1 / rank of first relevant document
    reciprocal_rank = 0.0
    for k, idx in enumerate(retrieved_contexts, 1):
        if idx in relevant_contexts:
            reciprocal_rank = 1.0 / k
            break

    all_recalls.append(recall)
    all_precisions.append(precision)
    all_avg_precisions.append(avg_precision)
    all_reciprocal_ranks.append(reciprocal_rank)

    # Check if answer contains relevant product info (faithfulness proxy)
    has_product_id = "ProductId:" in answer or "productid" in answer.lower()
    has_image = "ImageURL" in answer or "![" in answer
    faithfulness_proxy = 1.0 if (has_product_id and has_image) else 0.5

    results_list.append({
        "question": question,
        "answer": answer[:100] + "...",
        "ground_truth": ground_truth[:100] + "...",
        "recall": recall,
        "precision": precision,
        "avg_precision": avg_precision,
        "reciprocal_rank": reciprocal_rank,
        "faithfulness_proxy": faithfulness_proxy,
    })

results_df = pd.DataFrame(results_list)

# Calculate aggregate metrics
mean_recall = sum(all_recalls) / len(all_recalls) if all_recalls else 0.0
mean_precision = sum(all_precisions) / len(all_precisions) if all_precisions else 0.0
map_score = sum(all_avg_precisions) / len(all_avg_precisions) if all_avg_precisions else 0.0
mrr_score = sum(all_reciprocal_ranks) / len(all_reciprocal_ranks) if all_reciprocal_ranks else 0.0

print("\n" + "=" * 60)
print("Retrieval Evaluation Results")
print("=" * 60)
print(f"\n📊 Standard Retrieval Metrics:")
print(f"  • Recall (Completeness):           {mean_recall:.3f}")
print(f"  • Precision (Purity/Accuracy):     {mean_precision:.3f}")
print(f"  • MAP (Ranking Quality):           {map_score:.3f}")
print(f"  • MRR (Top Result):                {mrr_score:.3f}")
print(f"\n🤖 Answer Quality:")
print(f"  • Faithfulness Proxy:              {results_df['faithfulness_proxy'].mean():.3f}")
print("=" * 60)

print("\n📋 Per-example scores:")
display(results_df[[
    "question", "recall", "precision", "avg_precision",
    "reciprocal_rank", "faithfulness_proxy"
]])

Computing retrieval evaluation metrics...

Retrieval Evaluation Results

📊 Standard Retrieval Metrics:
  • Recall (Completeness):           1.000
  • Precision (Purity/Accuracy):     0.875
  • MAP (Ranking Quality):           1.000
  • MRR (Top Result):                1.000

🤖 Answer Quality:
  • Faithfulness Proxy:              1.000

📋 Per-example scores:


,question,recall,precision,avg_precision,reciprocal_rank,faithfulness_proxy
0,"Hey, do you have any white tops for girls from...",1.0,1.0,1.0,1.0,1.0
1,I'm looking for a black top for my daughter. D...,1.0,0.5,1.0,1.0,1.0
2,I'm looking for a blue casual top for my daugh...,1.0,1.0,1.0,1.0,1.0
3,"Hi, I'm looking for a cute pink top for my dau...",1.0,1.0,1.0,1.0,1.0


#### View Evaluation Results per example

In [14]:
# Display detailed results from retrieval evaluation
for i, row in results_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Example {i+1}:")
    print(f"Question: {row['question']}")
    print(f"\nGround Truth: {row['ground_truth']}")
    print(f"\nAgent Answer: {row['answer']}")
    print(f"\n📊 Retrieval Metrics:")
    print(f"  • Recall:           {row['recall']:.3f}  (completeness)")
    print(f"  • Precision:        {row['precision']:.3f}  (accuracy)")
    print(f"  • Avg Precision:    {row['avg_precision']:.3f}  (ranking)")
    print(f"  • Reciprocal Rank:  {row['reciprocal_rank']:.3f}  (top result)")
    print(f"\n🤖 Answer Quality:")
    print(f"  • Faithfulness:     {row['faithfulness_proxy']:.3f}")
    print(f"{'='*60}")


Example 1:
Question: Hey, do you have any white tops for girls from Gini and Jony in a large size?

Ground Truth: Yes, the Gini and Jony Girls Knit White Top (ProductId: 42419) is available in size Large. You can v...

Agent Answer: We have a few options for white tops from Gini and Jony for girls in a large size. Here are the prod...

📊 Retrieval Metrics:
  • Recall:           1.000  (completeness)
  • Precision:        1.000  (accuracy)
  • Avg Precision:    1.000  (ranking)
  • Reciprocal Rank:  1.000  (top result)

🤖 Answer Quality:
  • Faithfulness:     1.000

Example 2:
Question: I'm looking for a black top for my daughter. Do you have any options?

Ground Truth: Yes, we have the Gini and Jony Girls Black Top (ProductId: 34009). It's available in sizes Medium, X...

Agent Answer: We have a few options for black tops for girls in our catalog. Here are a few products that match yo...

📊 Retrieval Metrics:
  • Recall:           1.000  (completeness)
  • Precision:        0.500  (ac

#### Summary statistics from retrieval evaluation

In [ ]:
total = len(results_df)
mean_recall = results_df['recall'].mean()
mean_precision = results_df['precision'].mean()
map_score = results_df['avg_precision'].mean()
mrr_score = results_df['reciprocal_rank'].mean()
avg_faithfulness = results_df['faithfulness_proxy'].mean()

# Consider "passing" if metrics meet minimum thresholds
threshold_recall = 0.70      # 70% completeness
threshold_precision = 0.80   # 80% accuracy
threshold_faithfulness = 0.75  # 75% answer quality

passing = sum(1 for _, row in results_df.iterrows()
              if row['recall'] >= threshold_recall
              and row['precision'] >= threshold_precision
              and row['faithfulness_proxy'] >= threshold_faithfulness)

print(f"\n{'='*60}")
print(f"📊 RETRIEVAL EVALUATION SUMMARY")
print(f"{'='*60}")
print(f"\nTotal Examples Evaluated: {total}")
print(f"\n🎯 Standard Retrieval Metrics:")
print(f"  • Recall (Completeness):           {mean_recall:.3f}")
print(f"    → How many relevant docs were retrieved")
print(f"    → Use case: Medical/legal where missing docs is critical")
print(f"\n  • Precision (Purity/Accuracy):     {mean_precision:.3f}")
print(f"    → How many retrieved docs are relevant")
print(f"    → Use case: E-commerce where irrelevant results annoy users")
print(f"\n  • MAP (Ranking Quality):           {map_score:.3f}")
print(f"    → Quality of result ranking")
print(f"    → Use case: Search engines where order matters")
print(f"\n  • MRR (Top Result):                {mrr_score:.3f}")
print(f"    → Quality of first result")
print(f"    → Use case: Fact-finding where only first answer counts")
print(f"\n🤖 Answer Quality:")
print(f"  • Faithfulness Proxy:              {avg_faithfulness:.3f}")
print(f"    → Does answer include product details (ID, images)?")
print(f"\n✅ Pass Rate:")
print(f"  Examples meeting thresholds:")
print(f"    (recall≥{threshold_recall}, precision≥{threshold_precision}, faithfulness≥{threshold_faithfulness})")
print(f"  {passing}/{total} ({passing/total*100:.1f}%)")
print(f"{'='*60}")


📊 RETRIEVAL EVALUATION SUMMARY

Total Examples Evaluated: 4

🎯 Standard Retrieval Metrics:
  • Recall (Completeness):           1.000
    → How many relevant docs were retrieved
    → Use case: Medical/legal where missing docs is critical

  • Precision (Purity/Accuracy):     0.875
    → How many retrieved docs are relevant
    → Use case: E-commerce where irrelevant results annoy users

  • MAP (Ranking Quality):           1.000
    → Quality of result ranking
    → Use case: Search engines where order matters

  • MRR (Top Result):                1.000
    → Quality of first result
    → Use case: Fact-finding where only first answer counts

🤖 Answer Quality:
  • Faithfulness Proxy:              1.000
    → Does answer include product details (ID, images)?

✅ Pass Rate:
  Examples meeting thresholds:
    (recall≥0.7, precision≥0.8, faithfulness≥0.75)
  3/4 (75.0%)
